# Modules update
## Notebook to update current just-dna-seq modules

In [5]:
import polars as pl
import polars_bio as pb
import sqlite3
from pathlib import Path
from pycomfort import files

from platformdirs import user_cache_dir

# Configure Polars to show more rows and columns
pl.Config.set_tbl_rows(-1)  # Show all rows
pl.Config.set_tbl_cols(-1)  # Show all columns
pl.Config.set_tbl_width_chars(1000)  # Increase table width
pl.Config.set_fmt_str_lengths(1000)  # Show longer string values without truncation


polars.config.Config

In [6]:
from os import listdir
from prepare_annotations.resource import get_cache_dir
cache = get_cache_dir()
ensembl_cache = cache / "ensembl" / "homo_sapiens" 
listdir(ensembl_cache)

['homo_sapiens-chr2.parquet',
 'homo_sapiens-chr9.parquet',
 'homo_sapiens-chr17.parquet',
 'homo_sapiens-chr21.parquet',
 'homo_sapiens-chrY.parquet',
 'homo_sapiens-chr20.parquet',
 'homo_sapiens-chr19.parquet',
 'homo_sapiens-chr4.parquet',
 'homo_sapiens-chr14.parquet',
 'homo_sapiens-chr3.parquet',
 'homo_sapiens-chr13.parquet',
 'homo_sapiens-chrX.parquet',
 'homo_sapiens-chr6.parquet',
 'homo_sapiens-chr5.parquet',
 'homo_sapiens-chr22.parquet',
 'homo_sapiens-chr1.parquet',
 'homo_sapiens-chrMT.parquet',
 'vcf',
 'homo_sapiens-chr15.parquet',
 'homo_sapiens-chr16.tmp.parquet',
 'homo_sapiens-chr7.parquet',
 'homo_sapiens-chr12.parquet',
 'homo_sapiens-chr18.parquet',
 'homo_sapiens-chr8.tmp.parquet',
 'homo_sapiens-chr11.parquet',
 'homo_sapiens-chr10.parquet']

### Testing the parquets 

In [7]:
chr1 = pl.scan_parquet(ensembl_cache / "homo_sapiens-chr1.parquet")
chr1.head(5).collect()

chrom,start,end,id,ref,alt,alts,qual,filter,COSMIC_101,ClinVar_202502,dbSNP_156,HGMD-PUBLIC_20204,TSA,E_Cited,E_Multiple_observations,E_Freq,E_TOPMed,E_Hapmap,E_Phenotype_or_Disease,E_ESP,E_gnomAD,E_1000G,E_ExAC,CLIN_risk_factor,CLIN_protective,CLIN_confers_sensitivity,CLIN_other,CLIN_drug_response,CLIN_uncertain_significance,CLIN_benign,CLIN_likely_pathogenic,CLIN_pathogenic,CLIN_likely_benign,CLIN_histocompatibility,CLIN_not_provided,CLIN_association,MA,MAF,MAC,AA
str,u32,u32,str,str,str,list[str],f64,str,bool,bool,bool,bool,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,str,f32,i32,str
"""1""",10001,10001,"""rs1570391677""","""T""","""A|C""","[""A"", ""C""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10002,10002,"""rs1570391692""","""A""","""C""","[""C""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10003,10003,"""rs1570391694""","""A""","""C""","[""C""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10007,10007,"""rs1639538116""","""T""","""C|G""","[""C"", ""G""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10008,10008,"""rs1570391698""","""A""","""C|G|T""","[""C"", ""G"", ""T""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null


In [8]:
chr1_updated = chr1.with_columns(alts=pl.col("alt").str.split("|")).select(pl.col(chr1.collect_schema().names()[:chr1.collect_schema().names().index("alt")+1] + ["alts"] + chr1.collect_schema().names()[chr1.collect_schema().names().index("alt")+1:]))
chr1_updated.head(5).collect()

chrom,start,end,id,ref,alt,alts,qual,filter,COSMIC_101,ClinVar_202502,dbSNP_156,HGMD-PUBLIC_20204,TSA,E_Cited,E_Multiple_observations,E_Freq,E_TOPMed,E_Hapmap,E_Phenotype_or_Disease,E_ESP,E_gnomAD,E_1000G,E_ExAC,CLIN_risk_factor,CLIN_protective,CLIN_confers_sensitivity,CLIN_other,CLIN_drug_response,CLIN_uncertain_significance,CLIN_benign,CLIN_likely_pathogenic,CLIN_pathogenic,CLIN_likely_benign,CLIN_histocompatibility,CLIN_not_provided,CLIN_association,MA,MAF,MAC,AA
str,u32,u32,str,str,str,list[str],f64,str,bool,bool,bool,bool,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,str,f32,i32,str
"""1""",10001,10001,"""rs1570391677""","""T""","""A|C""","[""A"", ""C""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10002,10002,"""rs1570391692""","""A""","""C""","[""C""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10003,10003,"""rs1570391694""","""A""","""C""","[""C""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10007,10007,"""rs1639538116""","""T""","""C|G""","[""C"", ""G""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10008,10008,"""rs1570391698""","""A""","""C|G|T""","[""C"", ""G"", ""T""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null


In [11]:

chr1_updated.filter(pl.col("TSA")=="SNV").filter(pl.col("alts").list.len() > 2).head(5).collect()


chrom,start,end,id,ref,alt,alts,qual,filter,COSMIC_101,ClinVar_202502,dbSNP_156,HGMD-PUBLIC_20204,TSA,E_Cited,E_Multiple_observations,E_Freq,E_TOPMed,E_Hapmap,E_Phenotype_or_Disease,E_ESP,E_gnomAD,E_1000G,E_ExAC,CLIN_risk_factor,CLIN_protective,CLIN_confers_sensitivity,CLIN_other,CLIN_drug_response,CLIN_uncertain_significance,CLIN_benign,CLIN_likely_pathogenic,CLIN_pathogenic,CLIN_likely_benign,CLIN_histocompatibility,CLIN_not_provided,CLIN_association,MA,MAF,MAC,AA
str,u32,u32,str,str,str,list[str],f64,str,bool,bool,bool,bool,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,str,f32,i32,str
"""1""",10008,10008,"""rs1570391698""","""A""","""C|G|T""","[""C"", ""G"", ""T""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10014,10014,"""rs1639538207""","""A""","""C|G|T""","[""C"", ""G"", ""T""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10015,10015,"""rs1570391706""","""A""","""C|G|T""","[""C"", ""G"", ""T""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10020,10020,"""rs1570391708""","""A""","""C|G|T""","[""C"", ""G"", ""T""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null
"""1""",10027,10027,"""rs1570391716""","""A""","""C|G|T""","[""C"", ""G"", ""T""]",null,"""""",false,false,true,false,"""SNV""",false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,null


## Loading modules

In [12]:
from pycomfort import files
from pathlib import Path
base = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data = base / "data"
modules = data / "modules"
files.tprint(modules)







modules
	just_drugs
		annotation_tab.tsv
	just_longevitymap
		longevitymap.sqlite
	just_coronary
		coronary.sqlite
	just_lipidmetabolism
		lipid_metabolism.sqlite
	just_vo2max
		vo2max.sqlite
	just_superhuman
		superhuman.sqlite
	just_prs
		prs.sqlite
	just_cancer
		genes.txt


## Modules

In [13]:
import sqlite3
import polars as pl

# Connect to the database
db_path = modules / "just_longevitymap" / "longevitymap.sqlite"
conn = sqlite3.connect(db_path)

# Get list of all tables in the database
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables in the database:")
for table in tables:
    print(f"  - {table[0]}")



Tables in the database:
  - gene
  - population
  - variant
  - allele_weights
  - categories


In [14]:
# Get schema for each table
for table in tables:
    table_name = table[0]
    print(f"\nSchema for table '{table_name}':")
    cursor.execute(f"PRAGMA table_info({table_name});")
    columns = cursor.fetchall()
    for col in columns:
        print(f"  {col[1]} ({col[2]})")




Schema for table 'gene':
  id (INTEGER)
  name (TEXT)
  symbol (TEXT)
  alias (TEXT)
  description (TEXT)
  omim (TEXT)
  ensembl (TEXT)
  uniprot (TEXT)
  unigene (TEXT)
  cytogenetic_location (TEXT)

Schema for table 'population':
  id (INTEGER)
  name (TEXT)

Schema for table 'variant':
  id (INTEGER)
  location (TEXT)
  study_design (TEXT)
  conclusions (TEXT)
  association (TEXT)
  gender (TEXT)
  quickref (TEXT)
  quickyear (INTEGER)
  quickpubmed (TEXT)
  identifier (TEXT)
  gene_id (INTEGER)
  population_id (INTEGER)

Schema for table 'allele_weights':
  id (INTEGER)
  allele (TEXT)
  state (TEXT)
  zygosity (TEXT)
  weight (REAL)
  rsid (TEXT)
  priority (TEXT)
  category_id (INTEGER)

Schema for table 'categories':
  id (INTEGER)
  name (TEXT)


## Join with rsid

In [89]:
from prepare_annotations.resource import get_cache_dir
from pycomfort import files
cache = get_cache_dir()
ensembl_cache = cache / "ensembl_variations" / "splitted_variants"
files.tprint(ensembl_cache)


splitted_variants
	ensembl_variations.duckdb
	sequence_alteration
		homo_sapiens-chr2.parquet
		homo_sapiens-chr9.parquet
		homo_sapiens-chr17.parquet
		homo_sapiens-chr21.parquet
		homo_sapiens-chr8.parquet
		homo_sapiens-chrY.parquet
		homo_sapiens-chr20.parquet
		homo_sapiens-chr19.parquet
		homo_sapiens-chr4.parquet
		homo_sapiens-chr14.parquet
		homo_sapiens-chr3.parquet
		homo_sapiens-chr13.parquet
		homo_sapiens-chrX.parquet
		homo_sapiens-chr6.parquet
		homo_sapiens-chr5.parquet
		homo_sapiens-chr22.parquet
		homo_sapiens-chr1.parquet
		homo_sapiens-chr15.parquet
		homo_sapiens-chr16.parquet
		homo_sapiens-chr7.parquet
		homo_sapiens-chr12.parquet
		homo_sapiens-chr18.parquet
		homo_sapiens-chr11.parquet
		homo_sapiens-chr10.parquet
	SNV
		homo_sapiens-chr5.vortex
		homo_sapiens-chr2.vortex
		homo_sapiens-chr2.parquet
		homo_sapiens-chr9.parquet
		homo_sapiens-chr17.parquet
		homo_sapiens-chr16.vortex
		homo_sapiens-chr19.vortex
		homo_sapiens-chr10.vortex
		homo_sapiens-chr14.

In [7]:
weights = pl.read_database("SELECT id, allele, state, zygosity, weight, rsid, priority, category_id FROM allele_weights", connection=conn)
weights.head(5)

id,allele,state,zygosity,weight,rsid,priority,category_id
i64,str,str,str,f64,str,str,i64
0,"""T""","""alt""","""het""",0.5,"""rs7412""","""1.0""",1
1,"""T""","""alt""","""hom""",1.0,"""rs7412""","""1.0""",1
2,"""C""","""alt""","""het""",-0.5,"""rs429358""","""1.0""",1
3,"""C""","""alt""","""hom""",-1.0,"""rs429358""","""1.0""",1
4,"""G""","""ref""","""hom""",0.97,"""rs5882""","""0.97""",1


In [8]:
# Get unique rsids from small weights table
rsids = weights["rsid"].unique().to_list()
print(f"Filtering for {len(rsids)} unique rsids")


Filtering for 528 unique rsids


In [9]:
# Filter the large lazy frame first, then join
joined = df_ensembl.select(
    pl.col("chrom"), 
    pl.col("start"), 
    pl.col("end")
    ).filter(
    pl.col("id").is_in(rsids)
).join(
    weights.lazy(),
    left_on="id",
    right_on="rsid",
    how="inner"
)

In [10]:
example = joined.head(5).collect(engine="streaming")
example

ColumnNotFoundError: unable to find column "id"; valid columns: ["chrom", "start", "end"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
SELECT [col("chrom"), col("start"), col("end")]
  Parquet SCAN [/home/antonkulaga/.cache/just_dna_pipelines/ensembl_variations/splitted_variants/SNV/homo_sapiens-chr1.parquet, ... 24 other sources] [id: 129231707533808]
  PROJECT */40 COLUMNS

In [26]:
result = joined.collect(engine="streaming")
result

chrom,start,end,id,ref,alt,qual,filter,cosmic_101,clinvar_202502,dbsnp_156,hgmd-public_20204,tsa,e_cited,e_multiple_observations,e_freq,e_topmed,e_hapmap,e_phenotype_or_disease,e_esp,e_gnomad,e_1000g,e_exac,clin_risk_factor,clin_protective,clin_confers_sensitivity,clin_other,clin_drug_response,clin_uncertain_significance,clin_benign,clin_likely_pathogenic,clin_pathogenic,clin_likely_benign,clin_histocompatibility,clin_not_provided,clin_association,ma,maf,mac,aa,id_right,allele,state,zygosity,weight,priority,category_id
str,u32,u32,str,str,str,f64,str,bool,bool,bool,bool,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,str,f32,i32,str,i64,str,str,str,f64,str,i64
"""10""",67891367,67891367,"""rs7896005""","""A""","""G""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,"""A""",39,"""G""","""alt""","""het""",0.36,"""0.72""",5
"""10""",67891367,67891367,"""rs7896005""","""A""","""T""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,"""A""",39,"""G""","""alt""","""het""",0.36,"""0.72""",5
"""10""",67891367,67891367,"""rs7896005""","""A""","""G""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,"""A""",40,"""G""","""alt""","""hom""",0.72,"""0.72""",5
"""10""",67891367,67891367,"""rs7896005""","""A""","""T""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,"""A""",40,"""G""","""alt""","""hom""",0.72,"""0.72""",5
"""19""",44919689,44919689,"""rs4420638""","""A""","""G""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,true,false,true,true,false,false,false,false,false,false,false,false,false,false,false,false,true,false,"""G""",0.151099,770,"""G""",36,"""G""","""alt""","""het""",-0.375,"""0.75""",1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""12""",57774005,57774005,"""rs10877015""","""A""","""T""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,false,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,"""G""",0.355965,1814,"""A""",1009,"""C""","""alt""","""hom""",0.12,"""0.12""",11
"""22""",44203572,44203572,"""rs139170""","""C""","""G""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,false,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,"""T""",0.463305,2361,"""T""",990,"""T""","""alt""","""het""",0.065,"""0.13""",0
"""22""",44203572,44203572,"""rs139170""","""C""","""T""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,false,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,"""T""",0.463305,2361,"""T""",990,"""T""","""alt""","""het""",0.065,"""0.13""",0


In [33]:
weights.filter(pl.col("rsid") == "rs7896005")

id,allele,state,zygosity,weight,rsid,priority,category_id
i64,str,str,str,f64,str,str,i64
39,"""G""","""alt""","""het""",0.36,"""rs7896005""","""0.72""",5
40,"""G""","""alt""","""hom""",0.72,"""rs7896005""","""0.72""",5


In [31]:
result.filter(pl.col("id") == "rs7896005")

chrom,start,end,id,ref,alt,qual,filter,cosmic_101,clinvar_202502,dbsnp_156,hgmd-public_20204,tsa,e_cited,e_multiple_observations,e_freq,e_topmed,e_hapmap,e_phenotype_or_disease,e_esp,e_gnomad,e_1000g,e_exac,clin_risk_factor,clin_protective,clin_confers_sensitivity,clin_other,clin_drug_response,clin_uncertain_significance,clin_benign,clin_likely_pathogenic,clin_pathogenic,clin_likely_benign,clin_histocompatibility,clin_not_provided,clin_association,ma,maf,mac,aa,id_right,allele,state,zygosity,weight,priority,category_id
str,u32,u32,str,str,str,f64,str,bool,bool,bool,bool,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,str,f32,i32,str,i64,str,str,str,f64,str,i64
"""10""",67891367,67891367,"""rs7896005""","""A""","""G""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,"""A""",39,"""G""","""alt""","""het""",0.36,"""0.72""",5
"""10""",67891367,67891367,"""rs7896005""","""A""","""T""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,"""A""",39,"""G""","""alt""","""het""",0.36,"""0.72""",5
"""10""",67891367,67891367,"""rs7896005""","""A""","""G""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,"""A""",40,"""G""","""alt""","""hom""",0.72,"""0.72""",5
"""10""",67891367,67891367,"""rs7896005""","""A""","""T""",null,"""""",false,false,true,false,"""SNV""",true,false,true,true,false,false,true,true,true,true,false,false,false,false,false,false,false,false,false,false,false,false,false,null,null,null,"""A""",40,"""G""","""alt""","""hom""",0.72,"""0.72""",5


In [15]:
weights.shape

(1043, 8)

In [ ]:
# Check all tables for columns containing 'rsid' or 'rs' in the name
for table in tables:
    table_name = table[0]
    cursor.execute(f"PRAGMA table_info({table_name});")
    columns = cursor.fetchall()
    rsid_columns = [col[1] for col in columns if 'rs' in col[1].lower() or 'id' in col[1].lower()]
    if rsid_columns:
        print(f"\n{table_name}: {rsid_columns}")

